# 🏴‍☠️ NOVA RECOVERY: EL RESCATE
Usa este cuaderno si los anteriores fallaron. Está blindado contra desconexiones.

### Instrucciones críticas:
1. Sube `art_universe_500.jsonl`.
2. Ejecuta el primer bloque de Drive y **NO sigas hasta que diga 'Conectado'**.
3. Ejecuta todo lo demás.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/drive/MyDrive/Nova_Result_Recovery/', exist_ok=True)
print('✅ DRIVE LISTO. Los archivos se guardaran aqui obligatoriamente.')

In [ ]:
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import shutil

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.2-1b-instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

dataset = load_dataset("json", data_files="art_universe_500.jsonl", split="train")
def format_p(ex):
    return {"text": [f"### Instruction:\n{i}\n\n### Response:\n{o}" for i, o in zip(ex["instruction"], ex["output"])]}
dataset = dataset.map(format_p, batched = True)

model = FastLanguageModel.get_peft_model(model, r = 16, lora_alpha = 16, random_state = 3407)

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = dataset,
    dataset_text_field = "text", max_seq_length = 2048,
    args = TrainingArguments(max_steps = 100, learning_rate = 2e-4, output_dir = "outputs", logging_steps=10)
)
trainer.train()

print('💾 Fase Final: Exportando...')
model.save_pretrained_gguf("nova_final_brain", tokenizer, quantization_method = "q4_k_m")

import glob
gguf_file = glob.glob("*.gguf")[0]
print(f'✅ Guardando {gguf_file} en Drive...')
shutil.copy(gguf_file, f'/content/drive/MyDrive/Nova_Result_Recovery/{gguf_file}')
drive.flush_and_unmount()
print('🚀 ¡TERMINADO! El archivo esta seguro en tu Drive.')
from google.colab import files
files.download(gguf_file)